# Phase 2 — Pretraining and Evaluation (run once per language)

**Before running:**
1. Upload `kaggle_upload/code/` and `kaggle_upload/tokenizers/` (or the combined zip) as a Kaggle Dataset.
2. Attach that dataset, plus your existing `hindi-processed-dataset` / `nepali-processed-dataset`, as inputs to this notebook.
3. Turn on **GPU** (Settings → Accelerator) and **Internet** (only needed for the `pip install` cell).
4. In the `CONFIG` cell, set `LANGUAGE`. That's it — no dataset paths to edit; the setup cell auto-discovers everything from whatever's attached, regardless of Kaggle's exact mount path format or zip nesting.
5. Run all cells.

**If a session times out before training finishes** (Kaggle GPU sessions are time-limited): download the newest `checkpoint_step*.pt` from `/kaggle/working/<language>/checkpoints/`, upload it as a new Kaggle Dataset (or a new version of one), set `PREVIOUS_CHECKPOINT_DATASET` in the CONFIG cell to that dataset's path, and re-run the notebook — training resumes from that step automatically (see `Trainer.load_checkpoint`), it does not restart from scratch.

Run this notebook **twice total**: once with `LANGUAGE = "hindi"`, once with `LANGUAGE = "nepali"`. The two runs never share code state, data, or weights.

In [ ]:
# torch/numpy/matplotlib are preinstalled on Kaggle GPU notebooks;
# these three are not.
!pip install -q sentencepiece sacrebleu rouge-score

## CONFIG — edit this cell for your setup

In [ ]:
# ============================================================
LANGUAGE = "hindi"  # "hindi" or "nepali" -- the only thing you need to edit between runs

# Optional: path to a Kaggle dataset holding a checkpoint_step*.pt file
# from a previous session, to resume training across sessions. Leave as
# None for a fresh run (or for the first session).
PREVIOUS_CHECKPOINT_DATASET = None  # e.g. "/kaggle/input/hindi-checkpoint-v1"

WORK_DIR = f"/kaggle/working/{LANGUAGE}"
# ============================================================
# No dataset paths to configure below this line -- the setup cell
# auto-discovers the code, tokenizer, and data-split files from whatever
# datasets are attached, regardless of Kaggle's exact mount path format
# or how deeply nested your zip upload ended up.

assert LANGUAGE in ("hindi", "nepali")
print(f"Configured for: {LANGUAGE}")

## Setup: reconstruct the repo's directory layout under `/kaggle/working/`

The code (`hindi/train/prepare_data.py` etc.) resolves paths relative to its own file location, expecting `tokenizer/`, `data/splits/`, `model/`, `train/`, `eval/`, `configs/` as siblings — exactly the local repo's layout. Kaggle mounts the code, tokenizer, and data as three *separate* read-only input directories, so this cell links/copies them into one working directory matching that expected layout. No code changes needed — `prepare_data.py`, `dataset.py`, and `trainer.py` run completely unmodified, identically to running them locally.

In [ ]:
import os
import shutil
import sys

INPUT_ROOT = "/kaggle/input"

for sub in ["tokenizer", "data/splits", "data/tokenized", "checkpoints", "logs", "eval_outputs"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)


def find_file(search_root, filename):
    """Recursively search for `filename` under `search_root`. Safe to use
    for the tokenizer files because the filename itself already encodes
    the language (hindi_tokenizer.model vs. nepali_tokenizer.model), so a
    full-tree search under /kaggle/input is unambiguous even with both
    languages' datasets attached at once.
    """
    matches = []
    for dirpath, _, filenames in os.walk(search_root):
        if filename in filenames:
            matches.append(os.path.join(dirpath, filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find '{filename}' anywhere under {search_root}. "
            f"Make sure the right dataset is attached to this notebook (Input panel, top right)."
        )
    if len(matches) > 1:
        print(f"NOTE: multiple matches for '{filename}', using: {matches[0]}")
    return matches[0]


def find_dataset_root(keyword_parts):
    """Find the /kaggle/input dataset folder whose name contains every
    string in `keyword_parts` (case-insensitive) -- used to locate the
    right *language-specific* dataset even when multiple datasets are
    attached, since filenames like train.txt collide across languages
    but the dataset folder name (e.g. 'hindi-processed-dataset',
    'phase2-hindi-checkpoint-30000') does not. Robust to however Kaggle
    names/nests it (hyphens, underscores, an extra /datasets/<username>/
    path segment, etc.). Raises FileNotFoundError if nothing matches --
    callers that treat this as optional should catch that.
    """
    matches = []
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        for d in dirnames:
            name = d.lower()
            if all(k.lower() in name for k in keyword_parts):
                matches.append(os.path.join(dirpath, d))
    if not matches:
        raise FileNotFoundError(
            f"No attached dataset folder matching {keyword_parts} found under {INPUT_ROOT}."
        )
    matches.sort(key=len)  # shortest path = the dataset root itself, not a nested subfolder
    return matches[0]


def find_dir_ending_with(search_root, *path_parts):
    """Find a directory under `search_root` whose path ends with
    os.path.join(*path_parts) -- e.g. find_dir_ending_with(root, "hindi",
    "model") locates '.../hindi/model' regardless of how many parent
    folders (a zip's own top level, Kaggle's dataset-owner path segment,
    etc.) sit above it. Including the language name in the target suffix
    is what disambiguates Hindi's model/ folder from Nepali's when both
    are present in the same code dataset.
    """
    target = os.path.join(*path_parts)
    matches = []
    for dirpath, _, _ in os.walk(search_root):
        if dirpath.rstrip("/").endswith(target):
            matches.append(dirpath)
    if not matches:
        raise FileNotFoundError(f"Could not find a directory ending in '.../{target}' under {search_root}.")
    matches.sort(key=len)
    return matches[0]


def link_or_copy(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy(src, dst)


# Tokenizer files (Phase 1's own trained SentencePiece model -- never a
# pretrained tokenizer, never shared between languages).
tok_model = find_file(INPUT_ROOT, f"{LANGUAGE}_tokenizer.model")
tok_vocab = find_file(INPUT_ROOT, f"{LANGUAGE}_tokenizer.vocab")
link_or_copy(tok_model, os.path.join(WORK_DIR, "tokenizer", f"{LANGUAGE}_tokenizer.model"))
link_or_copy(tok_vocab, os.path.join(WORK_DIR, "tokenizer", f"{LANGUAGE}_tokenizer.vocab"))

# Phase 1 train/val/test splits -- locate THIS language's processed-data
# dataset by name first (train.txt alone would be ambiguous with both
# languages attached), then search only inside it.
data_root = find_dataset_root([LANGUAGE, "process"])
for split in ("train", "val", "test"):
    src = find_file(data_root, f"{split}.txt")
    link_or_copy(src, os.path.join(WORK_DIR, "data", "splits", f"{split}.txt"))

# Code: model/, train/, eval/, configs/ (copied, not symlinked, since
# nothing here is large and copies avoid any read-only surprises).
for sub in ("model", "train", "eval", "configs"):
    src_dir = find_dir_ending_with(INPUT_ROOT, LANGUAGE, sub)
    dst_dir = os.path.join(WORK_DIR, sub)
    if not os.path.exists(dst_dir):
        shutil.copytree(src_dir, dst_dir)

# Previous session's checkpoint, if one is attached. Auto-detected by
# dataset name containing this language + "checkpoint" (e.g. the
# "phase2-hindi-checkpoint-30000" dataset); set PREVIOUS_CHECKPOINT_DATASET
# above to force a specific path instead. If none is found/attached,
# this is a fresh run -- not an error.
import glob

if PREVIOUS_CHECKPOINT_DATASET:
    ckpt_root = PREVIOUS_CHECKPOINT_DATASET
else:
    try:
        ckpt_root = find_dataset_root([LANGUAGE, "checkpoint"])
    except FileNotFoundError:
        ckpt_root = None

if ckpt_root:
    prev_ckpts = glob.glob(os.path.join(ckpt_root, "**", "checkpoint_step*.pt"), recursive=True)
    for p in prev_ckpts:
        link_or_copy(p, os.path.join(WORK_DIR, "checkpoints", os.path.basename(p)))
    print(f"Found checkpoint dataset at {ckpt_root}: copied {len(prev_ckpts)} "
          f"checkpoint(s): {[os.path.basename(p) for p in prev_ckpts]}")
else:
    print("No previous-checkpoint dataset found/attached -- starting fresh.")

sys.path.insert(0, os.path.join(WORK_DIR, "model"))
sys.path.insert(0, os.path.join(WORK_DIR, "train"))
sys.path.insert(0, os.path.join(WORK_DIR, "eval"))

print(f"Setup complete. {LANGUAGE}/ layout under {WORK_DIR}:")
for sub in sorted(os.listdir(WORK_DIR)):
    print(" ", sub)

## Data preparation (Phase 2, tokenize Phase 1 splits into binary token arrays)

In [ ]:
from prepare_data import prepare

token_counts = prepare()
print(f"[{LANGUAGE}] tokenized counts: {token_counts}")

## Build the model (Phase 2, section 2.1) and report the exact parameter count

In [ ]:
import json

from gpt import GPTLanguageModel

with open(os.path.join(WORK_DIR, "configs", "model_config.json")) as f:
    model_cfg = json.load(f)

model = GPTLanguageModel(
    vocab_size=model_cfg["vocab_size"], d_model=model_cfg["d_model"], n_layers=model_cfg["n_layers"],
    n_heads=model_cfg["n_heads"], d_ff=model_cfg["d_ff"], context_length=model_cfg["context_length"],
    embed_dropout=model_cfg["embed_dropout"], attn_dropout=model_cfg["attn_dropout"],
    resid_dropout=model_cfg["resid_dropout"], tie_embeddings=model_cfg["tie_embeddings"],
)
n_params = model.num_parameters()
print(f"[{LANGUAGE}] exact trainable parameter count: {n_params:,} ({n_params/1e6:.3f}M)")
assert n_params == model_cfg["_num_parameters_verified"], "parameter count mismatch vs. the locally-verified config!"

## Pretraining (Phase 2, section 2.2)

AdamW + warmup/cosine LR schedule + periodic validation loss + checkpointing (weights, optimizer state, scheduler state, step, config). `resume=True` automatically continues from the newest checkpoint in `checkpoints/` if one exists (including one copied in above from a previous session) -- it does not need to be told to resume, only whether a checkpoint is present.

In [ ]:
import torch

from dataset import TokenBlockDataset
from trainer import Trainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected -- check Settings > Accelerator is set to GPU.")

with open(os.path.join(WORK_DIR, "configs", "training_config.json")) as f:
    train_cfg = json.load(f)

train_ds = TokenBlockDataset(os.path.join(WORK_DIR, "data", "tokenized", "train.bin"), train_cfg["context_length"])
val_ds = TokenBlockDataset(os.path.join(WORK_DIR, "data", "tokenized", "val.bin"), train_cfg["context_length"])
print(f"[{LANGUAGE}] train blocks: {len(train_ds):,}  val blocks: {len(val_ds):,}")

checkpoint_dir = os.path.join(WORK_DIR, "checkpoints")
trainer = Trainer(model, train_ds, val_ds, train_cfg, checkpoint_dir, device=device)
trainer.train(resume=True, log_csv_path=os.path.join(WORK_DIR, "logs", "train_log.csv"))
print(f"[{LANGUAGE}] stopped at step {trainer.step}/{train_cfg['max_steps']}")

## Plot the training/validation loss curve

In [ ]:
from plotting import plot_loss_curves

plot_loss_curves(
    trainer.loss_history, trainer.val_loss_history, LANGUAGE,
    os.path.join(WORK_DIR, "eval_outputs", "loss_curve.png"),
)
print("saved loss_curve.png")

## Evaluation (Phase 2, section 2.3) — only run once training has actually finished (`trainer.step == max_steps`); PPL/BPB and generation quality on a half-trained model are not meaningful.

In [ ]:
import sentencepiece as spm

from lm_metrics import compute_ppl_bpb

sp = spm.SentencePieceProcessor(model_file=os.path.join(WORK_DIR, "tokenizer", f"{LANGUAGE}_tokenizer.model"))

ppl_bpb = compute_ppl_bpb(
    model,
    os.path.join(WORK_DIR, "data", "tokenized", "val.bin"),
    os.path.join(WORK_DIR, "data", "splits", "val.txt"),
    context_length=model_cfg["context_length"],
    device=device,
)
print(f"[{LANGUAGE}] PPL/BPB: {json.dumps(ppl_bpb, indent=2)}")

In [ ]:
from generation_metrics import compute_bleu_chrf_rouge, diversity_diagnostics, generate_continuations

# Held-out prefixes and their reference continuations -- replace/extend
# with real held-out examples drawn from data/splits/test.txt before
# writing the final report; these are placeholders to exercise the
# pipeline end to end.
if LANGUAGE == "hindi":
    prefixes = ["\u092d\u093e\u0930\u0924 \u090f\u0915 \u0935\u093f\u0936\u093e\u0932 \u0926\u0947\u0936 \u0939\u0948", "\u092a\u094d\u0930\u0927\u093e\u0928\u092e\u0902\u0924\u094d\u0930\u0940 \u0928\u0947 \u0906\u091c"]
    references = ["\u092d\u093e\u0930\u0924 \u090f\u0915 \u0935\u093f\u0936\u093e\u0932 \u0926\u0947\u0936 \u0939\u0948 \u091c\u0939\u093e\u0901 \u0905\u0928\u0947\u0915 \u092d\u093e\u0937\u093e\u090f\u0901 \u092c\u094b\u0932\u0940 \u091c\u093e\u0924\u0940 \u0939\u0948\u0902\u0964", "\u092a\u094d\u0930\u0927\u093e\u0928\u092e\u0902\u0924\u094d\u0930\u0940 \u0928\u0947 \u0906\u091c \u0928\u0908 \u0926\u093f\u0932\u094d\u0932\u0940 \u092e\u0947\u0902 \u0935\u093f\u0938\u094d\u0924\u093e\u0930 \u0938\u0947 \u092c\u0948\u0920\u0915 \u0915\u0940\u0964"]
else:
    prefixes = ["\u0928\u0947\u092a\u093e\u0932 \u0939\u093f\u092e\u093e\u0932\u092f\u0915\u094b \u0915\u093e\u0916\u092e\u093e", "\u092a\u094d\u0930\u0927\u093e\u0928\u092e\u0928\u094d\u0924\u094d\u0930\u0940\u0932\u0947 \u0906\u091c"]
    references = ["\u0928\u0947\u092a\u093e\u0932 \u0939\u093f\u092e\u093e\u0932\u092f\u0915\u094b \u0915\u093e\u0916\u092e\u093e \u0930\u0939\u0947\u0915\u094b \u0938\u0941\u0928\u094d\u0926\u0930 \u0926\u0947\u0936 \u0939\u094b\u0964", "\u092a\u094d\u0930\u0927\u093e\u0928\u092e\u0928\u094d\u0924\u094d\u0930\u0940\u0932\u0947 \u0906\u091c \u0915\u093e\u0920\u092e\u093e\u0921\u094c\u0902\u092e\u093e \u092c\u0948\u0920\u0915 \u0917\u0930\u094d\u0928\u0941\u092d\u092f\u094b\u0964"]

all_generations = {}
for label, kwargs in [("greedy", dict(greedy=True)),
                       ("temp_0.5", dict(temperature=0.5)),
                       ("temp_1.0", dict(temperature=1.0)),
                       ("temp_1.5", dict(temperature=1.5))]:
    gens = generate_continuations(model, sp, prefixes, max_new_tokens=40, device=device, **kwargs)
    all_generations[label] = gens
    print(f"--- {label} ---")
    for p, g in zip(prefixes, gens):
        print(f"  prefix: {p}\n  output: {g}\n")

bcr = compute_bleu_chrf_rouge(all_generations["greedy"], references)
diversity = diversity_diagnostics([g for gens in all_generations.values() for g in gens])
print(f"[{LANGUAGE}] BLEU/chrF/ROUGE-L (greedy vs. references): {json.dumps(bcr, indent=2)}")
print(f"[{LANGUAGE}] diversity diagnostics: {json.dumps(diversity, indent=2)}")

In [ ]:
from attention_analysis import attention_entropy, get_attention_weights, mean_attention_distance
from plotting import plot_attention_heatmap, plot_distance_grid, plot_entropy_grid

example_sentence = prefixes[0]
pieces, attn_per_layer = get_attention_weights(model, sp, example_sentence, device=device)

entropy_grid = attention_entropy(attn_per_layer)
distance_grid = mean_attention_distance(attn_per_layer)
plot_entropy_grid(entropy_grid, LANGUAGE, os.path.join(WORK_DIR, "eval_outputs", "attention_entropy.png"))
plot_distance_grid(distance_grid, LANGUAGE, os.path.join(WORK_DIR, "eval_outputs", "attention_distance.png"))

# Required: at least one early layer, one late layer, multiple heads.
n_layers = len(attn_per_layer)
n_heads = attn_per_layer[0].shape[0]
for layer_idx in (0, n_layers - 1):
    for head_idx in range(min(2, n_heads)):
        plot_attention_heatmap(
            attn_per_layer[layer_idx][head_idx].numpy(), pieces, layer_idx, head_idx, LANGUAGE,
            os.path.join(WORK_DIR, "eval_outputs", f"attn_heatmap_layer{layer_idx}_head{head_idx}.png"),
        )
print(f"[{LANGUAGE}] attention figures saved to {WORK_DIR}/eval_outputs/")

## Save a results summary and the checkpoint for download

In [ ]:
summary = {
    "language": LANGUAGE,
    "num_parameters": n_params,
    "final_step": trainer.step,
    "max_steps": train_cfg["max_steps"],
    "ppl_bpb": ppl_bpb,
    "bleu_chrf_rouge": bcr,
    "diversity": diversity,
    "attention_entropy_grid": entropy_grid.tolist(),
    "attention_distance_grid": distance_grid.tolist(),
}
with open(os.path.join(WORK_DIR, "eval_outputs", "results_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"[{LANGUAGE}] done. Download from /kaggle/working/{LANGUAGE}/:")
print("  - checkpoints/checkpoint_step*.pt  (latest = final trained weights)")
print("  - logs/train_log.csv, logs/train_log_val.csv")
print("  - eval_outputs/  (results_summary.json + all PNG figures)")